# Road Accidents Comparison: United States vs United Kingdom

## Introduction
This notebook compares road accident patterns, severity, temporal trends, and contributing factors between the US and UK.

**Objectives:**
- Analyze temporal patterns (year, month, hour, day of week)
- Compare severity distributions
- Explore weather, road conditions, and location factors
- Identify similarities and key differences
- Provide data-driven insights

**Datasets:**
- **US**: US Accidents
- **UK**: UK Road Safety Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os
from urllib.request import urlretrieve
import zipfile

# import geopandas as gpd
# import folium

%matplotlib inline


In [ ]:

# UK Collisions data
uk_collision_url = "https://data.dft.gov.uk/road-accidents-safety-data/dft-road-casualty-statistics-collision-last-5-years.csv"

print("Downloading UK Collisions data...")
uk_acc = pd.read_csv(uk_collision_url, low_memory=False)

uk_acc['collision_year'].dtype
years = uk_acc['collision_year'].unique()
years.sort()

print("UK Collisions Shape:", uk_acc.shape)
print("Columns:", uk_acc.columns.tolist())
print("Years covered:", years)

UK Collisions Shape: (503475, 44)
Columns: ['collision_index', 'collision_year', 'collision_ref_no', 'location_easting_osgr', 'location_northing_osgr', 'longitude', 'latitude', 'police_force', 'collision_severity', 'number_of_vehicles', 'number_of_casualties', 'date', 'day_of_week', 'time', 'local_authority_district', 'local_authority_ons_district', 'local_authority_highway', 'local_authority_highway_current', 'first_road_class', 'first_road_number', 'road_type', 'speed_limit', 'junction_detail_historic', 'junction_detail', 'junction_control', 'second_road_class', 'second_road_number', 'pedestrian_crossing_human_control_historic', 'pedestrian_crossing_physical_facilities_historic', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards_historic', 'carriageway_hazards', 'urban_or_rural_area', 'did_police_officer_attend_scene_of_accident', 'trunk_road_flag', 'lsoa_of_accident_location', 'enhanced_seve

In [ ]:


# Direct download link for the official 500k sample
url = "https://drive.google.com/uc?id=1U3u8QYzLjnEaSurtZfSAS_oh9AT2Mn8X&export=download"
zip_path = "US_Accidents_sample_500k.zip"
csv_path = "US_Accidents_March23_sampled.csv"

print("Downloading US Accidents sample (500k rows)... This may take 20-60 seconds.")

# Download the zip file
urlretrieve(url, zip_path)

# Extract the CSV
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")
    # The extracted file name might be US_Accidents_March23.csv or similar
    print("Files extracted:", os.listdir("."))

# Load the data
us_df = pd.read_csv(csv_path, low_memory=False)

print("US Sample loaded successfully!")
print("Shape:", us_df.shape)
print("Years:", sorted(us_df['Start_Time'].str[:4].unique() if 'Start_Time' in us_df.columns else "N/A"))

In [ ]:
def preprocess_uk_official(df):
    df = df.copy()
    
    # Rename key columns for easier use
    df.rename(columns={
        'collision_year': 'Year',
        'collision_severity': 'Severity',           # 1=Fatal, 2=Serious, 3=Slight
        'date': 'Date',
        'time': 'Time',
        'latitude': 'Latitude',
        'longitude': 'Longitude',
        'day_of_week': 'Day_of_Week',
        'speed_limit': 'Speed_Limit',
        'urban_or_rural_area': 'Urban_Rural'
    }, inplace=True)
    
    # Convert date and time
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce').dt.hour
    
    # Create categorical severity labels for better readability
    severity_map = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
    df['Severity_Label'] = df['Severity'].map(severity_map)
    
    return df

uk_clean = preprocess_uk_official(uk_acc)
print(uk_clean.head())

  collision_index  Year collision_ref_no  location_easting_osgr  \
0   2021170H10421  2021        170H10421               447098.0   
1   2021170H11231  2021        170H11231               450486.0   
2   2020170M11750  2020        170M11750               449694.0   
3   2021170M31761  2021        170M31761               449744.0   
4   2021170S10441  2021        170S10441               445971.0   

   location_northing_osgr  Longitude   Latitude  police_force  Severity  \
0                532997.0  -1.270905  54.689833            17         3   
1                533118.0  -1.218333  54.690592            17         3   
2                519733.0  -1.232884  54.570397            17         3   
3                514217.0  -1.233040  54.520825            17         3   
4                520834.0  -1.290292  54.580641            17         3   

   number_of_vehicles  ...  Urban_Rural  \
0                   2  ...            2   
1                   2  ...            1   
2                

/tmp/ipykernel_15725/1694064775.py:18: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
